# 02 · Feature Engineering & Scaling
**16 leakage-free features:** seasonality (2) + lagged NDVI (5) + weather (6) + demography (3). Lags use `shift(1)` so no future leakage. Scaler fit on train only.


In [ ]:
from pathlib import Path
import json, joblib
import pandas as pd, numpy as np
from sklearn.preprocessing import StandardScaler

DATA_DIR = Path("../../data_pipeline/data/processed")
MODEL_DIR = Path("../models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Load train/test with fallback
if (DATA_DIR / "master_train_2015_2023.csv").exists():
    train_df = pd.read_csv(DATA_DIR / "master_train_2015_2023.csv")
    test_df = pd.read_csv(DATA_DIR / "master_test_2024_2025.csv")
else:
    base = pd.read_csv(DATA_DIR / "ml_base_2015_2025.csv")
    train_df = base[base["year"]<=2023].copy()
    test_df = base[base["year"]>=2024].copy()

print(f"Train {train_df.shape} | Test {test_df.shape}")
display(train_df.head(2))


In [ ]:
# Combine to compute lags across 2023->2024 boundary
full = pd.concat([train_df.assign(_split="train"), test_df.assign(_split="test")], ignore_index=True)
full = full.sort_values(["county","year","month"]).reset_index(drop=True)

# 1. Seasonality (MAM & OND rainy seasons)
full["month_sin"] = np.sin(2*np.pi*full["month"]/12).round(6)
full["month_cos"] = np.cos(2*np.pi*full["month"]/12).round(6)

# 2. Lagged vegetation (strictly past via shift)
g = full.groupby("county")["ndvi_mean"]
full["ndvi_lag1"] = g.shift(1)
full["ndvi_lag3"] = g.shift(3)
full["ndvi_roll3_mean"] = g.transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())
full["ndvi_roll3_std"] = g.transform(lambda s: s.shift(1).rolling(3, min_periods=1).std()).fillna(0)
for c in ["ndvi_lag1","ndvi_lag3","ndvi_roll3_mean"]:
    full[c] = full[c].fillna(full["ndvi_mean_hist"])
full["ndvi_anomaly"] = (full["ndvi_lag1"] - full["ndvi_mean_hist"]).round(6)

# 3. Weather (proposal 3.3.2: rainfall anomaly + forage dryness)
full["rain_3m_sum"] = full.groupby("county")["rainfall_mm"].transform(lambda s: s.rolling(3, min_periods=1).sum()).round(4)
full["soil_deficit"] = (0.45 - full["soil_moisture"]).clip(lower=0).round(4)
ndvi_drop = (full["ndvi_mean_hist"] - full["ndvi_lag1"]).clip(lower=0) / (full["ndvi_std_hist"] + 1e-6)
full["forage_dryness_index"] = (0.5*ndvi_drop + 0.5*full["soil_deficit"]*full["temp_mean_c"]/25).round(4)

# 4. Demography (KNBS 2019, static)
full["farming_ratio"] = (full["farming"]/(full["total"]+1)).round(4)
full["livestock_share"] = (full["livestock_production"]/(full["crop_production"]+full["livestock_production"]+1)).round(4)
full["irrigation_ratio"] = (full["irrigation"]/(full["farming"]+1)).round(4)

train_fe = full[full["_split"]=="train"].drop(columns=["_split"]).reset_index(drop=True)
test_fe = full[full["_split"]=="test"].drop(columns=["_split"]).reset_index(drop=True)
print(f"✓ Features built: train {train_fe.shape} test {test_fe.shape}")


In [ ]:
# 16 leakage-free features (current-month ndvi_mean excluded - it defines target)
FEATURE_COLS = [
    "month_sin","month_cos",
    "ndvi_lag1","ndvi_lag3","ndvi_roll3_mean","ndvi_roll3_std","ndvi_anomaly",
    "temp_mean_c","rainfall_mm","soil_moisture","rainfall_anomaly",
    "rain_3m_sum","forage_dryness_index",
    "farming_ratio","livestock_share","irrigation_ratio",
]

X_train = train_fe[FEATURE_COLS]
y_train = train_fe["target_risk"]
X_test = test_fe[FEATURE_COLS]
y_test = test_fe["target_risk"]

assert X_train.isna().sum().sum()==0 and X_test.isna().sum().sum()==0, "NA in features"

scaler = StandardScaler()
scaler.fit(X_train)

# Save
for df, name in [(X_train,"X_train"),(y_train.to_frame(),"y_train"),(X_test,"X_test"),(y_test.to_frame(),"y_test")]:
    df.to_csv(DATA_DIR / f"{name}.csv", index=False)
train_fe.to_csv(DATA_DIR / "train_matrix.csv", index=False)
test_fe.to_csv(DATA_DIR / "test_matrix.csv", index=False)

with open(MODEL_DIR / "feature_columns.json","w") as f:
    json.dump(FEATURE_COLS, f, indent=2)
joblib.dump(scaler, MODEL_DIR / "scaler.joblib")
print(f"✓ Saved {len(FEATURE_COLS)} features | X_train {X_train.shape} X_test {X_test.shape}")
display(pd.DataFrame({"feature":FEATURE_COLS}))
